# Data Reconnaissance: AI Content Decay Analysis

**Research question:** What is the gap between AI product release velocity and public learning-content coverage/update frequency?

**Practical question:** Can we collect enough public data to measure this responsibly?

This notebook documents what data is available, what is missing, and what can be measured honestly.

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

INTERIM = Path('../data/interim')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Source Inventory

| Source | Vendor | Type | URL |
|--------|--------|------|-----|
| Claude Release Notes | Anthropic | product_release | support.claude.com |
| Claude Code Changelog | Anthropic | product_release | code.claude.com |
| Gemini API Changelog | Google | product_release | ai.google.dev |
| OpenAI Changelog | OpenAI | product_release | platform.openai.com |
| OpenAI Index | OpenAI | product_release | openai.com/index |
| Anthropic Academy | Anthropic | learning_content | anthropic.skilljar.com |
| OpenAI Academy | OpenAI | learning_content | academy.openai.com |
| Google AI Tutorials | Google | learning_content | ai.google.dev |

## 2. Fetch Status

In [ ]:
status = pd.read_csv(INTERIM / 'source_status.csv')
status[['source_id', 'vendor', 'source_type', 'fetch_method', 'fetch_status', 'http_status', 'notes']]

In [ ]:
# Fetch success rate
total = len(status)
success = status['fetch_status'].str.startswith('success').sum()
print(f'Fetch success rate: {success}/{total} ({success/total:.0%})')
print(f'\nFailed sources:')
failed = status[~status['fetch_status'].str.startswith('success')]
for _, row in failed.iterrows():
    print(f'  {row["source_id"]}: {row["fetch_status"]} (HTTP {row["http_status"]})')

## 3. Release Data Availability

In [ ]:
releases = pd.read_csv(INTERIM / 'product_releases.csv')
print(f'Total release entries: {len(releases)}')
print()
print('Row counts by vendor:')
print(releases.groupby('vendor').size().to_frame('count'))
print()
print('Row counts by vendor + product_area:')
print(releases.groupby(['vendor', 'product_area']).size().to_frame('count'))

In [ ]:
# Topic category distribution
print('Topic categories by vendor:')
ct = pd.crosstab(releases['vendor'], releases['topic_category'])
ct

In [ ]:
# Date range and coverage
releases['release_date'] = pd.to_datetime(releases['release_date'])
print('Date ranges by vendor:')
for vendor in releases['vendor'].unique():
    v = releases[releases['vendor'] == vendor]
    print(f'  {vendor}: {v["release_date"].min().date()} to {v["release_date"].max().date()} ({len(v)} entries)')

In [ ]:
# Release velocity by month
releases['month'] = releases['release_date'].dt.to_period('M')
monthly = releases.groupby(['vendor', 'month']).size().unstack('vendor', fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
monthly.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Release Entries per Month by Vendor')
ax.set_ylabel('Count')
ax.set_xlabel('Month')
# Show every 3rd label to reduce clutter
labels = ax.get_xticklabels()
for i, label in enumerate(labels):
    if i % 3 != 0:
        label.set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../outputs/charts/release_velocity_by_month.png', dpi=150)
plt.show()

In [ ]:
# Breaking changes
breaking = releases[releases['is_breaking_change'] == True]
print(f'Breaking changes / deprecations: {len(breaking)}')
print(breaking.groupby('vendor').size().to_frame('count'))

## 4. Learning Content Data Availability

In [ ]:
learning = pd.read_csv(INTERIM / 'learning_content.csv')
print(f'Total learning content entries: {len(learning)}')
print()
print('Row counts by vendor:')
print(learning.groupby('vendor').size().to_frame('count'))
print()
print('Topic categories by vendor:')
print(pd.crosstab(learning['vendor'], learning['topic_category']))

In [ ]:
# Date visibility — the critical question
has_date = learning['visible_last_updated'].notna() & (learning['visible_last_updated'] != '')
print('Update date visibility by vendor:')
for vendor in learning['vendor'].unique():
    v_mask = learning['vendor'] == vendor
    total_v = v_mask.sum()
    with_date = (v_mask & has_date).sum()
    print(f'  {vendor}: {with_date}/{total_v} have visible update dates')
print()
print(f'OVERALL: {has_date.sum()}/{len(learning)} entries have visible update dates')
print()
if has_date.sum() == 0:
    print('>>> FINDING: No learning platform exposes course update timestamps publicly.')
    print('>>> The metric must be COVERAGE GAP, not UPDATE LAG.')

In [ ]:
# Anthropic Academy courses (primary learning source)
anthropic_courses = learning[learning['vendor'] == 'Anthropic'][['title', 'topic_category', 'visible_last_updated', 'notes']]
print(f'Anthropic Academy: {len(anthropic_courses)} courses')
anthropic_courses

## 5. What Can Be Measured Honestly

Based on data reconnaissance:

**Measurable (with available data):**
- Release count per month by vendor (Anthropic, Google)
- Release-to-coverage status: does a course exist that covers a given release topic?
- Uncovered major feature count: releases with no matching course content
- Topic category coverage: which categories have courses vs which do not
- Content decay risk by topic: high-velocity categories with low course coverage
- Stale-risk score: based on release velocity and missing/partial course coverage

**NOT measurable (data not available):**
- Exact update lag (release date → course update date) — update timestamps not public
- Course quality or depth
- Course completion rates
- OpenAI release velocity (API returns 403)

## 6. What Cannot Be Measured from Public Data

| Gap | Reason | Impact |
|-----|--------|--------|
| Course update timestamps | Not exposed by Skilljar, OpenAI Academy, or Google | Cannot measure exact update lag |
| OpenAI release data | platform.openai.com returns 403 | Cannot include OpenAI in v1 |
| Course internal content | Would require enrollment | Cannot verify content depth |
| Content freshness | No last-modified dates | Must use coverage proxy instead |

## 7. Fields Available by Source

### Product Release Sources

| Field | Anthropic Releases | Claude Code | Google Gemini | OpenAI |
|-------|-------------------|-------------|---------------|--------|
| release_date | Yes | Yes | Yes | N/A (403) |
| title | Yes | Yes (version) | Yes | N/A |
| description | Yes | Yes (items) | Yes | N/A |
| topic_category | Derived | Derived | Derived | N/A |
| is_breaking_change | Derived | Derived | Derived | N/A |

### Learning Content Sources

| Field | Anthropic Academy | OpenAI Academy | Google Tutorials |
|-------|------------------|----------------|------------------|
| title | Yes | Noisy (nav items) | Noisy (nav items) |
| description | Yes | No | No |
| url | Yes | Partial | Partial |
| visible_last_updated | **No** | **No** | **No** |
| topic_category | Derived | Low quality | Low quality |

## 8. Reliability Score by Source

In [ ]:
reliability = pd.DataFrame([
    {'source': 'Anthropic Release Notes', 'vendor': 'Anthropic', 'type': 'product_release',
     'reliability': 'high', 'rows': 37, 'date_quality': 'exact', 'notes': 'Clean structured dates and titles'},
    {'source': 'Claude Code Changelog', 'vendor': 'Anthropic', 'type': 'product_release',
     'reliability': 'high', 'rows': 292, 'date_quality': 'exact', 'notes': 'Versioned entries with dates'},
    {'source': 'Google Gemini Changelog', 'vendor': 'Google', 'type': 'product_release',
     'reliability': 'high', 'rows': 243, 'date_quality': 'exact', 'notes': 'Structured dated entries'},
    {'source': 'OpenAI Changelog', 'vendor': 'OpenAI', 'type': 'product_release',
     'reliability': 'not_usable', 'rows': 0, 'date_quality': 'n/a', 'notes': 'HTTP 403 - blocked'},
    {'source': 'OpenAI Index', 'vendor': 'OpenAI', 'type': 'product_release',
     'reliability': 'not_usable', 'rows': 0, 'date_quality': 'n/a', 'notes': 'HTTP 403 - blocked'},
    {'source': 'Anthropic Academy', 'vendor': 'Anthropic', 'type': 'learning_content',
     'reliability': 'high', 'rows': 18, 'date_quality': 'none', 'notes': 'Clean course catalog, no update dates'},
    {'source': 'OpenAI Academy', 'vendor': 'OpenAI', 'type': 'learning_content',
     'reliability': 'needs_review', 'rows': 19, 'date_quality': 'none', 'notes': 'Contains nav items, needs manual cleanup'},
    {'source': 'Google AI Tutorials', 'vendor': 'Google', 'type': 'learning_content',
     'reliability': 'needs_review', 'rows': 122, 'date_quality': 'none', 'notes': 'Quickstart page, heavy nav noise'},
])
reliability

## 9. Recommended Final Notebook Scope

### Metric
**Coverage gap** (not update lag)

Since no learning platform exposes course update timestamps, the honest metric is:

> *AI product release velocity vs. public learning-content coverage as of collection date*

### Vendors for v1
- **Anthropic** (primary): Clean release data + clean course catalog
- **Google** (secondary): Clean release data, learning content needs manual review
- **OpenAI** (excluded from v1): Both release and learning sources blocked or noisy

### Recommended analyses
1. Release count per month (Anthropic, Google)
2. Release-to-coverage status by topic category
3. Uncovered major feature count
4. Content decay risk score by topic
5. Stale-risk heatmap: release velocity × coverage gap

### Effort estimate
- **3 days** if scoped to Anthropic-only analysis
- **5 days** if including Google with manual learning content cleanup
- **7 days** if attempting OpenAI workarounds